# Swiss Law Legal Information Retrieval - Baseline Baseline

## 1. Written Description of the Competition Task
The goal of this competition is to build an information retrieval system for Swiss law. Given a legal question formulated in **English**, the objective is to retrieve a semicolon-separated list of the most relevant Swiss legal sources (statutes, federal court decisions, etc.), which are primarily written in **German, French, or Italian**.

Legal research is highly specific; therefore, the task requires exact matches of canonical legal citations. The system operates under realistic production constraints, including a bounded 12-hour compute time and a strictly offline execution environment (no internet access during inference).

## 2. Dataset Overview and Evaluation Metric

### Dataset Overview
The dataset consists of multiple files:
* **`train.csv`**: Public training queries (in non-English languages) paired with gold-standard citation labels. Based on the LEXam dataset.
* **`val.csv`**: A small validation set of 10 English queries with gold citations.
* **`test.csv`**: The hidden test set (English queries). This notebook will generate predictions for these IDs.
* **`laws_de.csv`**: The retrieval corpus of Swiss federal law snippets in German, keyed by a canonical citation string.
* **`court_considerations.csv`**: A massive retrieval corpus of Swiss Federal Court decisions. *(Note: Handled in future iterations due to compute size).*

### Evaluation Metric
The competition evaluates submissions using the **Macro F1 Score** computed at the citation level. 
* **True Positives:** Citations predicted by the model that exactly match the strings in the gold citation set.
* **Precision:** How many of the predicted citations were correct.
* **Recall:** How many of the actual gold citations were successfully retrieved.
The F1 score is calculated per-query and then averaged across all queries. This metric penalizes systems that guess too many irrelevant citations (poor precision) or miss vital ones (poor recall).

In [1]:
import re
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm

# Deep Learning Libraries
import torch
from sentence_transformers import SentenceTransformer, util

# Set device to GPU if available for faster Deep Learning inference
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [6]:
# =========================
# LOAD DATA & EDA
# =========================
print("Loading data...")
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")
laws_df  = pd.read_csv("laws_de.csv")

print(f"Train set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"Laws corpus shape: {laws_df.shape}")

# Auto-detect columns based on Kaggle schema
QUERY_COL = "query" if "query" in train_df.columns else "question"
LABEL_COL = "gold_citations" if "gold_citations" in train_df.columns else "labels"
LAW_COL   = "citation" if "citation" in laws_df.columns else laws_df.columns[0]

display(train_df.head(2))

Loading data...
Train set shape: (1139, 3)
Validation set shape: (10, 3)
Test set shape: (40, 2)
Laws corpus shape: (175933, 3)


,query_id,query,gold_citations
0,train_0001,Die A AG betreibt seit den 1970er-Jahren auf d...,Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10...
1,train_0002,Die Alpha Consulting AG kann nun ihr Grundstüc...,Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;...


In [7]:
def parse_labels(x):
    if pd.isna(x): return []
    return [c.strip() for c in str(x).split(";") if c.strip()]

train_df["cit_list"] = train_df[LABEL_COL].apply(parse_labels)
val_df["cit_list"]   = val_df[LABEL_COL].apply(parse_labels)

# EDA: How many citations are typically expected?
train_df['citation_count'] = train_df['cit_list'].apply(len)
print("\n--- EDA: Citation Count Distribution in Train ---")
print(train_df['citation_count'].describe())

# Create our retrievable corpus
corpus_citations = laws_df[LAW_COL].astype(str).tolist()
corpus_texts = laws_df['text'].astype(str).tolist()


--- EDA: Citation Count Distribution in Train ---
count    1139.000000
mean        4.090430
std         4.512206
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        44.000000
Name: citation_count, dtype: float64


## 3. Deep Learning Baseline Model
Because the queries are in English but the law corpus is largely in German, traditional keyword matching (like TF-IDF) struggles with the language barrier. 

To solve this, we implement a **Deep Learning Semantic Search** approach using a Multilingual Sentence Transformer (`paraphrase-multilingual-MiniLM-L12-v2`). This neural network embeds both English questions and German legal texts into the same dense vector space, allowing us to compute Cosine Similarity between them.

In [8]:
# Initialize the Multilingual Deep Learning Model
# NOTE FOR KAGGLE SUBMISSION: You must download this model, upload it as a Kaggle dataset, 
# and replace the string below with your local offline path (e.g., '/kaggle/input/minilm-model')
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
print(f"Loading Deep Learning Model: {model_name}...")
embedder = SentenceTransformer(model_name, device=device)

# Encode the corpus (this takes time, so we wrap it in a progress bar)
print("Encoding Law Corpus into Dense Vectors...")
corpus_embeddings = embedder.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True)

Loading Deep Learning Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...
Encoding Law Corpus into Dense Vectors...


Batches:   0%|          | 0/5498 [00:00<?, ?it/s]

In [9]:
# Regex for explicit article mentions (Heuristic rule)
ART_PATTERN = re.compile(r'art\.?\s*(\d+[a-z]*)\s*(?:abs\.?\s*(\d+))?\s*([a-z]{2,10})', re.IGNORECASE)

def extract_regex(query):
    matches = ART_PATTERN.findall(query)
    results = []
    for art, abs_, law in matches:
        if abs_: c = f"Art. {art} Abs. {abs_} {law.upper()}"
        else:    c = f"Art. {art} {law.upper()}"
        results.append(c)
    return results

def predict_citations(query, top_k=20):
    # 1. Deep Learning Semantic Search
    query_embedding = embedder.encode(query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
    
    # Get top 50 semantic hits
    top_results = torch.topk(cos_scores, k=50)
    
    scores = Counter()
    
    # Add neural network scores to our counter
    for score, idx in zip(top_results[0], top_results[1]):
        citation_id = corpus_citations[idx]
        scores[citation_id] += score.item()
        
    # 2. Heuristic Regex Boost (If the user explicitly types an article)
    regex_hits = extract_regex(query)
    for c in regex_hits:
        scores[c] += 2.0  # Massive boost for explicit exact regex matches

    # Return the top_k ranked citations
    ranked = [c for c, _ in scores.most_common(top_k)]
    return ranked

In [10]:
# Helper to normalize strings for fair F1 comparison
def normalize(x):
    return re.sub(r'[^a-z0-9]', '', x.lower())

def calculate_f1(pred, gold):
    pred_set = set(map(normalize, pred))
    gold_set = set(map(normalize, gold))

    if not pred_set or not gold_set: return 0
    tp = len(pred_set & gold_set)
    precision = tp / len(pred_set)
    recall = tp / len(gold_set)

    if precision + recall == 0: return 0
    return 2 * precision * recall / (precision + recall)

# Run validation
print("\nEvaluating on Validation Set...")
val_scores = []

for _, row in val_df.iterrows():
    q = row[QUERY_COL]
    gold = row["cit_list"]
    
    pred = predict_citations(q, top_k=5) # Returning top 5 based on our EDA average
    
    f1 = calculate_f1(pred, gold)
    val_scores.append(f1)

# Documented Performance Score
baseline_score = np.mean(val_scores)
print(f"\n🔥 DOCUMENTED BASELINE MACRO F1 SCORE: {baseline_score:.4f} 🔥")


Evaluating on Validation Set...

🔥 DOCUMENTED BASELINE MACRO F1 SCORE: 0.0363 🔥


In [11]:
print("\nPredicting on Test Set for Kaggle Submission...")
results = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred = predict_citations(row[QUERY_COL], top_k=5)
    
    results.append({
        "query_id": row["query_id"],
        "predicted_citations": ";".join(pred)
    })

submission = pd.DataFrame(results)
submission.to_csv("submission.csv", index=False)

print("\n✅ Submission saved successfully to submission.csv!")
display(submission.head())


Predicting on Test Set for Kaggle Submission...


100%|██████████| 40/40 [00:01<00:00, 28.32it/s]


✅ Submission saved successfully to submission.csv!


,query_id,predicted_citations
0,test_001,Art. 6 Abs. 1 946.231.143.6;Art. 10 Abs. 3 VTE...
1,test_002,Art. 83 SVG;Art. 59 Abs. 1 SVG;Art. 19 Abs. 3 ...
2,test_003,Art. 267a CO;Art. 11 Abs. 1 STUV;Art. 3 Abs. 1...
3,test_004,Art. 166 Abs. 2 SCHKG;Art. 37e Abs. 7 KLV;Art....
4,test_005,Art. 74 GBV;Art. 28 Abs. 2 AVV;Art. 68 Abs. 1 ...
